# Movie Ratings Dataset - Statistics Notebook

## 1. Load & Explore the Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

mydata = pd.read_csv("movie_ratings.csv")

In [ ]:
mydata.head()

In [ ]:
mydata.tail(5)

In [ ]:
mydata.info()

In [ ]:
mydata.describe()

## 2. Histograms

In [ ]:
mydata[['Rating', 'Age']].hist(figsize=(12, 5))
plt.suptitle('Histograms of Numeric Features', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
mydata.hist(by='Genre', column='Rating', figsize=(16, 10))
plt.suptitle('Rating Distribution by Genre', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
mydata.hist(by='Gender', column='Rating', figsize=(10, 5))
plt.suptitle('Rating Distribution by Gender', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
mydata.hist(by='Genre', column='Age', figsize=(16, 10))
plt.suptitle('Age Distribution by Genre', fontsize=14)
plt.tight_layout()
plt.show()

## 3. Cross-tabulation & Pivot Tables

In [ ]:
pd.crosstab(mydata['Genre'], mydata['Rating'].apply(lambda x: 'high' if x >= 4 else 'low'), margins=True)

In [ ]:
mydata['age_group'] = mydata['Age'].apply(lambda x: 'senior' if x >= 40 else ('adult' if x >= 25 else 'young'))
mydata['rating_group'] = mydata['Rating'].apply(lambda x: 'high' if x >= 4 else 'low')
pd.pivot_table(mydata, index=['Genre', 'age_group'], columns=['rating_group'], aggfunc=len, fill_value=0)

In [ ]:
pd.pivot_table(mydata, values='Rating',
              index=['Genre'],
              columns=['Gender'],
              aggfunc='mean').round(2)

## 4. Box Plots & Outlier Detection

In [ ]:
sns.boxplot(x='Gender', y='Rating', data=mydata)
plt.title('Rating by Gender')
plt.show()

In [ ]:
q1, q3 = np.percentile(mydata['Rating'], [25, 75])
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr
outliers = mydata[(mydata['Rating'] < lower) | (mydata['Rating'] > upper)]
print(f'Q1: {q1:.4f}, Q3: {q3:.4f}, IQR: {iqr:.4f}')
print(f'Lower bound: {lower:.4f}, Upper bound: {upper:.4f}')
print(f'Number of outliers: {len(outliers)}')

In [ ]:
plt.figure(figsize=(8, 5))
plt.boxplot(mydata['Rating'], vert=False, patch_artist=True,
            boxprops=dict(facecolor='lightgreen', color='black'),
            flierprops=dict(marker='o', markerfacecolor='red', markersize=8))
plt.title('Box Plot: Rating')
plt.xlabel('Rating')
plt.show()

## 5. Count Plots

In [ ]:
sns.countplot(x='Genre', hue='rating_group', data=mydata)
plt.title('High vs Low Ratings by Genre')
plt.xticks(rotation=30)
plt.show()

## 6. Measures of Central Tendency & Dispersion

In [ ]:
mean_val = np.mean(mydata['Rating'])
std_val  = np.std(mydata['Rating'])
print(f'Mean Rating : {mean_val:.4f}')
print(f'Std  Rating : {std_val:.4f}')

In [ ]:
q1, q2, q3 = np.percentile(mydata['Rating'], [25, 50, 75])
print(f'Q1: {q1:.4f}  Median (Q2): {q2:.4f}  Q3: {q3:.4f}')

In [ ]:
print(mydata.mean(numeric_only=True))

In [ ]:
print(mydata.std(numeric_only=True))

## 7. Skewness & Distribution Shapes

In [ ]:
from scipy import stats as st

normal_dist = np.random.normal(mean_val, std_val, 500)
skewed_dist = st.skewnorm.rvs(a=10, loc=mean_val, scale=std_val, size=500)

plt.figure(figsize=(10, 5))
plt.hist(normal_dist, bins=30, alpha=0.5, label='Normal', color='skyblue')
plt.hist(skewed_dist, bins=30, alpha=0.5, label='Right Skewed', color='salmon')
plt.title('Normal vs Right-Skewed Distribution')
plt.legend()
plt.show()

In [ ]:
fig, ax = plt.subplots()
plt.axvline(x=mydata['Rating'].mean(), color='orange', label='Mean')
plt.axvline(x=np.median(mydata['Rating']), color='green', label='Median')
plt.hist(mydata['Rating'], color='lightgray', bins=5)
plt.title('Rating: Mean vs Median')
plt.legend()
plt.show()

## 8. Z-Scores & p-Values

In [ ]:
z_scores = st.zscore(mydata['Rating'])
print('First 10 z-scores:', z_scores[:10].values)

In [ ]:
z_input = 1.5
x_val = mean_val + z_input * std_val
print(f'Raw score for Z={z_input}: {x_val:.4f}')

In [ ]:
prob = st.norm.cdf(3, loc=mean_val, scale=std_val)
print(f'P(Rating < 3): {prob:.4f}')

p_below = st.norm.cdf(-2.5)
p_above = 1 - st.norm.cdf(2.5)
print(f'P-value (|Z| > 2.5): {p_below + p_above:.4f}')

## 9. Chebyshev's Rule

In [ ]:
def chebyshev_rule(k):
    if k <= 1:
        return 'k must be greater than 1'
    return (1 - (1 / k**2)) * 100

print(f'At least {chebyshev_rule(2):.2f}% within 2 SD')
print(f'At least {chebyshev_rule(3):.2f}% within 3 SD')

within_k2 = mydata[(mydata['Rating'] >= mean_val - 2*std_val) &
                   (mydata['Rating'] <= mean_val + 2*std_val)]
print(f'Actual within 2 SD: {100*len(within_k2)/len(mydata):.2f}%')

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(mydata['Rating'], bins=5, color='gray', alpha=0.4)
plt.axvline(mean_val, color='blue', linestyle='dashed', label='Mean')
plt.axvline(mean_val - 2*std_val, color='green', linestyle='--', label='k=2 Lower')
plt.axvline(mean_val + 2*std_val, color='green', linestyle='--', label='k=2 Upper')
plt.title("Chebyshev's Theorem – Rating")
plt.legend()
plt.show()

## 10. t-Tests

In [ ]:
null_mean = 3.0
t_stat, p_val = st.ttest_1samp(mydata['Rating'], null_mean)
print(f'T-statistic: {t_stat:.4f}')
print(f'P-value    : {p_val:.4f}')
alpha = 0.05
if p_val < alpha:
    print('Decision: Reject H0 – mean rating differs significantly from', null_mean)
else:
    print('Decision: Fail to reject H0')

In [ ]:
male_ratings   = mydata[mydata['Gender'] == 'Male']['Rating'].values
female_ratings = mydata[mydata['Gender'] == 'Female']['Rating'].values
t2, p2 = st.ttest_ind(male_ratings, female_ratings, equal_var=False)
print(f'Male mean rating  : {male_ratings.mean():.4f}')
print(f'Female mean rating: {female_ratings.mean():.4f}')
print(f'T-statistic: {t2:.4f},  P-value: {p2:.4f}')

## 11. Confidence Intervals

In [ ]:
n    = mydata['Rating'].size
s    = mydata['Rating'].std()
xbar = mydata['Rating'].mean()
z    = 1.96

CI_err = z * (s / n**0.5)
print(f'95% CI: ({xbar - CI_err:.4f}, {xbar + CI_err:.4f})')

fig, ax = plt.subplots()
plt.ylabel('Rating')
plt.grid(axis='y')
ax.errorbar(['Rating'], [xbar], [CI_err], fmt='o', color='green')
plt.title('95% Confidence Interval for Mean Rating')
plt.show()

## 12. Correlation Analysis

In [ ]:
corr = mydata.corr(numeric_only=True)
corr

In [ ]:
plt.figure(figsize=(6, 5))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Correlation Heatmap')
plt.show()

In [ ]:
r, p = st.pearsonr(mydata['Age'], mydata['Rating'])
print(f'Pearson r = {r:.4f},  p-value = {p:.4e}')
print(f'R-squared = {r**2:.4f}')

## 13. Scatter Plots & Pair Plot

In [ ]:
sns.scatterplot(x='Age', y='Rating', hue='Gender', data=mydata)
plt.title('Age vs Rating by Gender')
plt.show()

In [ ]:
sns.pairplot(mydata[['Age', 'Rating', 'Gender']], hue='Gender')
plt.show()

## 14. Distribution Plots

In [ ]:
sns.displot(mydata['Rating'], kde=True)
plt.title('Distribution of Rating')
plt.show()

In [ ]:
sns.displot(mydata['Age'], kde=True)
plt.title('Distribution of Age')
plt.show()

## 15. Linear Regression (OLS)

In [ ]:
x = mydata['Age'].values
y = mydata['Rating'].values

cov_mat = np.cov(x, y)
beta1 = cov_mat[0, 1] / cov_mat[0, 0]
beta0 = y.mean() - beta1 * x.mean()
print(f'Intercept (β0): {beta0:.4f}')
print(f'Slope     (β1): {beta1:.4f}')

In [ ]:
xline = np.linspace(x.min(), x.max(), 1000)
yline = beta0 + beta1 * xline

sns.scatterplot(x=x, y=y, alpha=0.5)
plt.plot(xline, yline, color='orange')
plt.xlabel('Age')
plt.ylabel('Rating')
plt.title('Linear Regression: Age → Rating')
plt.show()

In [ ]:
import statsmodels.api as sm

X_ols = sm.add_constant(mydata[['Age']])
model = sm.OLS(mydata['Rating'], X_ols)
result = model.fit()
print(result.summary())